Este módulo permite realizar web scraping sobre la red social X (anteriormente Twitter).
Está compuesto por dos componentes principales:

1. **Recolección por palabras clave:** Obtiene publicaciones a partir de un conjunto
   de términos definidos por el usuario, lo que facilita la captura de contenido
   relacionado con una temática específica.

2. **Recokección por cuentas:** Obtiene publicaciones de usuarios previamente seleccionados.

In [ ]:
import os
import time
import pandas as pd
import tweepy
from datetime import datetime

# API de X

Configuración de la API para adquisición de datos de redes sociales. 

Esta compontente está ajustada por motivos de seguridad, es meramente ilustrativa para mostrar como se adquieren los datos, pero por motivos de licencias no puedo compartir los tokens.

In [ ]:
# Configura tu token de acceso
bearer_token = ''
client = tweepy.Client(bearer_token=bearer_token)

# Lista de usuarios


usuarios = ['pereira_sa6822'] 


# Función para obtener tweets con geolocalización incluida
def obtener_ultimos_tweets(usuario, fecha_inicio, fecha_fin, max_tweets):
    tweet_data = []  # Almacena los datos de cada tweet
    
    try:
        # Obtener el ID de usuario basado en el nombre de usuario
        user = client.get_user(username=usuario)
        user_id = user.data.id

        # Convertir fechas a formato ISO
        fecha_inicio_iso = datetime.strptime(fecha_inicio, "%Y-%m-%d").isoformat() + "Z"
        fecha_fin_iso = datetime.strptime(fecha_fin, "%Y-%m-%d").isoformat() + "Z"

        # Obtener los tweets dentro del rango de fechas con campos adicionales, incluyendo geolocalización
        tweets = client.get_users_tweets(
            id=user_id,
            max_results=max_tweets,
            start_time=fecha_inicio_iso,
            end_time=fecha_fin_iso,
            tweet_fields=['created_at', 'public_metrics', 'entities', 'geo']
        )

        if tweets.data:
            for tweet in tweets.data:
                metrics = tweet.public_metrics
                enlaces = []

                # Comprobar si el tweet tiene enlaces
                if tweet.entities and 'urls' in tweet.entities:
                    enlaces = [url['expanded_url'] for url in tweet.entities['urls']]

                # Extraer información de geolocalización, si está disponible
                geo_info = tweet.geo.get('place_id') if tweet.geo else None

                # Agregar detalles del tweet a la lista
                tweet_data.append({
                    "Usuario": usuario,
                    "Nombre": user.data.name,
                    "Fecha": tweet.created_at,
                    "Texto": tweet.text,
                    "Likes": metrics['like_count'],
                    "Retweets": metrics['retweet_count'],
                    "Comentarios": metrics['reply_count'],
                    "Citas": metrics['quote_count'],
                    "Enlaces": ", ".join(enlaces),
                    "Geolocalización": geo_info  
                })

    except tweepy.TooManyRequests:
        print(f"Demasiadas solicitudes para {usuario}, esperando 15 minutos...")
        time.sleep(900)  # Esperar 15 minutos antes de continuar
    except Exception as e:
        print(f"Error con {usuario}: {e}")
    
    return tweet_data

# Lista para almacenar los datos de todos los tweets
todos_los_tweets = []

# Rango de fechas y cantidad de tweets deseados
fecha_inicio = "2024-11-03"
fecha_fin = "2024-11-10"
max_tweets = 6

# Obtener tweets de cada usuario
for usuario in usuarios:
    tweets_usuario = obtener_ultimos_tweets(usuario, fecha_inicio, fecha_fin, max_tweets)
    todos_los_tweets.extend(tweets_usuario)
    
    # Pausa de 1.5 segundos entre solicitudes para evitar exceder el límite de la API
    time.sleep(2)

# Guardar todos los datos en un DataFrame y luego a un CSV
df = pd.DataFrame(todos_los_tweets)

# Crear la carpeta en el directorio actual si no existe
directorio_actual = os.getcwd()
nombre_carpeta = os.path.join(directorio_actual, "/Users/Daniel/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/GIES_OCI/WebScraping_prueba")
os.makedirs(nombre_carpeta, exist_ok=True)

# Definir la ruta completa para guardar el archivo
nombre_archivo = os.path.join(nombre_carpeta, "prueba.csv")
df.to_csv(nombre_archivo, index=False, encoding="utf-8")
print(f"Datos guardados en {nombre_archivo}")

# Bajando datos por cuentas preseleccionadas

In [ ]:
# Configura tu token de acceso
bearer_token = ''
client = tweepy.Client(bearer_token=bearer_token)

# Lista de usuarios


# usuarios = [
#     'Cicmty', 'conagua_mx', 'elnortelocal', 'LoQuePasaenNL', 'eldelaconsty', 'camion_desde',
#     'metrorreynlofi1', 'MovilidadSN', 'SSP_Apodaca', 'PC_NuevoLeon', 'TodosSomosHLM',
#     'AtentosMTYSur', 'Apodaca_News', 'ArbSanJorge', 'SoyDeSanNicolas', 'abimaelsal',
#     'BomberosNL', 'nelvaldez', 'MAGS_SP', 'saz2000', 'DarkKnightMty', 'revistacodigo21',
#     'Mty_Leones', 'cesarmty', '911SanPedro', 'SSPCMonterrey', 'pc_mty', 'rayelizalder',
#     'Informativo3651', 'arualsanpedrogg', 'JulioCesarCano', 'nmasmonterrey', 'info7mty',
#     'ABCNoticiasMX', 'AsiEsMonterrey', 'GobSanNicolas', 'PortalElPunto',
#     'gob_Escobed', 'mtygob', 'joluisgarcia', 'visionmty1', 'QuePasaEnS', 'LCAlatorre',
#     'IdentidadNL', 'MtyFollow', 'ayd_monterrey', 'ComunidadMTY', 'municipiodegpe',
#     'CiudaDanaMtySur', 'MilenioMty', 'telediariomty', 'webcamsdemexico', 'antinundacion'
# ]


usuarios = [ 'elnorte', 'Milenio', 'Reforma', 'samuel_garcias', 'nmas', 'MilenioMty', 'telediariomty',
             'webcamsdemexico', 'antinundacion', 'Formula_Mty', 'TraficoenMty', 'escuadronvial', 'ABCNoticiasMX',
             'PC_NuevoLeon', 'RvTallerbeto', 'AleGon_27', 'nuevoleon', 'RcNL_', 'LCAlatorre', 'rayelizalder', 
             'MAGS_SP', 'NuevoLeonAlerta', 'QuePasaEnSN', 'NotidigitalNL', 'MtyFollowMx', 'MtyFollow', 'camion_desde',
             'conagua_clima', 'meteoredmx', 'AztecaNoticias', 'enriqueburgosv', 'InfoMeteoro', 'GobSanNicolas', 'revistacodigo21',
             'yadithvaldez', 'DarkKnightMty', 'AsiEsMonterrey', 'lopezdoriga', 'LupitaJuarezH', 'azucenau', 'TitularNoticias',
             'MVSNoticiasMty', 'mexnewz'
             ] 


# Función para obtener tweets con geolocalización incluida
def obtener_ultimos_tweets(usuario, fecha_inicio, fecha_fin, max_tweets):
    tweet_data = []  # Almacena los datos de cada tweet
    
    try:
        # Obtener el ID de usuario basado en el nombre de usuario
        user = client.get_user(username=usuario)
        user_id = user.data.id

        # Convertir fechas a formato ISO
        fecha_inicio_iso = datetime.strptime(fecha_inicio, "%Y-%m-%d").isoformat() + "Z"
        fecha_fin_iso = datetime.strptime(fecha_fin, "%Y-%m-%d").isoformat() + "Z"

        # Obtener los tweets dentro del rango de fechas con campos adicionales, incluyendo geolocalización
        tweets = client.get_users_tweets(
            id=user_id,
            max_results=max_tweets,
            start_time=fecha_inicio_iso,
            end_time=fecha_fin_iso,
            tweet_fields=['created_at', 'public_metrics', 'entities', 'geo']
        )

        if tweets.data:
            for tweet in tweets.data:
                metrics = tweet.public_metrics
                enlaces = []

                # Comprobar si el tweet tiene enlaces
                if tweet.entities and 'urls' in tweet.entities:
                    enlaces = [url['expanded_url'] for url in tweet.entities['urls']]

                # Extraer información de geolocalización, si está disponible
                geo_info = tweet.geo.get('place_id') if tweet.geo else None

                # Agregar detalles del tweet a la lista
                tweet_data.append({
                    "Usuario": usuario,
                    "Nombre": user.data.name,
                    "Fecha": tweet.created_at,
                    "Texto": tweet.text,
                    "Likes": metrics['like_count'],
                    "Retweets": metrics['retweet_count'],
                    "Comentarios": metrics['reply_count'],
                    "Citas": metrics['quote_count'],
                    "Enlaces": ", ".join(enlaces),
                    "Geolocalización": geo_info  
                })

    except tweepy.TooManyRequests:
        print(f"Demasiadas solicitudes para {usuario}, esperando 15 minutos...")
        time.sleep(900)  # Esperar 15 minutos antes de continuar
    except Exception as e:
        print(f"Error con {usuario}: {e}")
    
    return tweet_data

# Lista para almacenar los datos de todos los tweets
todos_los_tweets = []

# Rango de fechas y cantidad de tweets deseados
fecha_inicio = "2021-10-14"
fecha_fin = "2021-10-18"
max_tweets = 50

# Obtener tweets de cada usuario
for usuario in usuarios:
    tweets_usuario = obtener_ultimos_tweets(usuario, fecha_inicio, fecha_fin, max_tweets)
    todos_los_tweets.extend(tweets_usuario)
    
    # Pausa de 1.5 segundos entre solicitudes para evitar exceder el límite de la API
    time.sleep(2)

# Guardar todos los datos en un DataFrame y luego a un CSV
df = pd.DataFrame(todos_los_tweets)

# Crear la carpeta en el directorio actual si no existe
directorio_actual = os.getcwd()
nombre_carpeta = os.path.join(directorio_actual, "/Users/Daniel/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Luisa Fernanda Chaparro Sierra's files - Analisis Tweets/Checkpoint Semestre Ago-Dic 2024/Web Scrapping/Web Scraping con API May 2025")
os.makedirs(nombre_carpeta, exist_ok=True)

# Definir la ruta completa para guardar el archivo
nombre_archivo = os.path.join(nombre_carpeta, "tweets_may_2025_1.csv")
df.to_csv(nombre_archivo, index=False, encoding="utf-8")
print(f"Datos guardados en {nombre_archivo}")